# Identifier les textes évoquant le concept de république

In [36]:
import pandas as pd
import re

# TODO: aviser si vire id_orateur et utiliser id_acteur partout
df = pd.read_csv(
    "../data/interim/data_cleaning_interv_regrouped.csv",
    low_memory=False,
    dtype={"ID_orateur": str},
)
df.shape

(425561, 56)

## INTRODUIRE PRÉ-TRAITEMENT TEXTE

Pour simplifier la vie et faciliter aussi possibles perf d'un futur modèle, virer les parenthèses et balises ici pour s'économiser pas mal de choses du côté des noms de groupes

In [ ]:
# nettoyage basique du texte
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["texte_brut"] = df["texte"]  # garder une version brute du texte
df["texte"] = df["texte"].apply(nettoyer_texte)

# Si pas géré avant (devrait le faire dans le 2 sinon, mais possible perte avec nettoyage)
# Supprimer les lignes où "texte" est manquant
# pas déconnant de le garder là avec éventuelles suppressions dues au nettoyage
df = df.dropna(subset=["texte"])
df.shape

## Regex

Logique de la tentative :
- regex
- mais exclure certains termes
- mais comme les termes exclus peuvent apparaitre aussi avec les termes voulus, éviter de chainer et finir par virer des trucs qu'on aurait voulu (les idées républicaines sont menacées par Les Républicains)

In [ ]:
# préparer les pays à exclure
with open("../data/raw/liste_pays_republique_stable.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays, plus bas on rendra le groupe non capturant
pattern_pays = r"|".join(re.escape(p) for p in liste_pays)


# Regex du champ lexical République (simplifié ici)

pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure
# logique : création de groupes (?:…) non capturant
# car utilisé juste pour les positions, pas besoin de les récupérer

# Expressions à exclure - casse exacte
# possible cas du féminin… mais pas d'occurrences dans la base avec nos exclusions
pattern_excl_case_sensitive = re.compile(
    r"(?:\b[LlDd]es Républicains\b)"  # garde la casse pour identifier le parti (et pas un adjectif)
    r"|(?:\baux Républicains\b)"  # idem majuscule pour le groupe
    r"|(?:\bsénateurs? Républicains?\b)"  # pas de féminin dans la base après exclu, mais aviser
    r"|(?:\bdéputés? Républicains?\b)"  # pas de féminin dans la base après exclu, mais aviser
)

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # partis et groupes politiques
    r"(?:\bgauche démocrate et républicaine\b)"  # premier sans |
    r"|(?:\brépublique en marche\b)"
    r"|(?:\bsocialiste, écologiste et républicain\b)"
    # fonctions et institutions
    r"|(?:\bprésident[s]? de la république\b)"
    r"|(?:\bprésidence[s]? de la république\b)"
    r"|(?:\bprocureur[s]? de la république\b)"
    r"|(?:\bcour[s]? de justice de la république\b)"
    r"|(?:\bcour[s]? de sûreté de la république\b)"
    r"|(?:\badministration générale de la république\b)"
    r"|(?:\bgouvernement de la république française\b)"
    # expression et législations
    r"|(?:\bcontrat d[’']engagement républicain\b)"  # à aviser
    # pays
    r"|(?:\brépublique[s]? soviétique[s]?\b)"  # pas un pays mais des expressions : aviser
    r"|(?:\brépublique[s]? de Weimar\b)"
    r"|(?:\b(?:" + pattern_pays + r")\b)",  # ajout des exclusions de pays
    re.I,
)


def contains_lexical_outside_excl(text):
    # TODO : aviser si veut utiliser spans triés et bisect pour optimiser la vérification des positions,
    # ou si pas besoin (en fonction du nombre d'exclusions et de la longueur des textes)
    # et on pourrait merger les positions de spans d'exclusions pour accélérer ?
    # mais pas indispensable ici et plus compliqué ? 


    # si pas de match lexical inutile d'aller plus loin
    if not pattern_lexical.search(text):
        return False

    # Collecter les spans exclus
    # en ajoutant les exclusions sensibles et insensibles à la casse
    excl_positions = [m.span() for m in pattern_excl_case_sensitive.finditer(text)] + [
        m.span() for m in pattern_excl_case_insensitive.finditer(text)
    ]

    # Fonction pour vérifier si une position est dans une zone exclue
    # optimisable avec bisect si besoin
    def in_excl(pos):
        # return any(start <= pos < end for start, end in excl_positions) # equivalent mais moins clair
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurrences du champ lexical
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False


In [42]:
# bloc d'essai
mon_texte = "Les Députés Républicains ont voté une loi."  # devrait être exclu
contains_lexical_outside_excl(mon_texte)


True

In [43]:
# Appliquer sur la colonne
df["repu_match_valide"] = df["texte"].apply(contains_lexical_outside_excl)

In [44]:
df["repu_match_valide"].value_counts()

repu_match_valide
False    414228
True      11317
Name: count, dtype: int64

In [ ]:
# trace matthias, aviser avec lui :)
# # Exporter fichier avec 2 colonnes pour calcul avec proportions
df.to_csv(
    "../data/interim/df_regroup_repu_proportion.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

In [46]:
df_match = df[df["repu_match_valide"]]
df_match.shape

(11317, 58)

In [47]:
df_match

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,active,dateMaj,dateSeance_ts,parti_affiliation,parti_affiliation_bis,texte_brut,repu_match_valide
18,CRSANR5L15S2017O1N125,NaN,NaN,20170628150000000,mercredi 28 juin 2017,Unique,125,AN,15,Session ordinaire 2016-2017,...,0.06,0.951,0.342,0.0,2025-06-24,2017-06-28 15:00:00,LR,LR,"Monsieur le président, la majorité a décidé de...",True
50,CRSANR5L15S2017O1N001,NaN,NaN,20170703150000000,lundi 03 juillet 2017,Unique,1,AN,15,Congrès du Parlement du 3 juillet 2017,...,NaN,NaN,NaN,NaN,NaN,2017-07-03 15:00:00,NaN,NaN,"Monsieur le président du Congrès, monsieur le ...",True
51,CRSJOCGR5L15S2017E1N001,NaN,NaN,20170703150000000,lundi 03 juillet 2017,Unique,1,AN,15,Congrès du Parlement du 3 juillet 2017,...,NaN,NaN,NaN,NaN,NaN,2017-07-03 15:00:00,NaN,NaN,"Monsieur le président du Congrès, monsieur le ...",True
52,CRSANR5L15S2017O1N001,NaN,NaN,20170703150000000,lundi 03 juillet 2017,Unique,1,AN,15,Congrès du Parlement du 3 juillet 2017,...,0.52,0.994,0.994,0.0,2025-06-24,2017-07-03 15:00:00,REN,REN,"Monsieur le président du Congrès, monsieur le ...",True
53,CRSJOCGR5L15S2017E1N001,NaN,NaN,20170703150000000,lundi 03 juillet 2017,Unique,1,AN,15,Congrès du Parlement du 3 juillet 2017,...,0.52,0.994,0.994,0.0,2025-06-24,2017-07-03 15:00:00,REN,REN,"Monsieur le président du Congrès, monsieur le ...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
424330,CRSANR5L16S2024O1N230,RUANR5L16S2024IDS28423,SCR5A2024O1,20240606090000000,jeudi 06 juin 2024,1,230,AN,16,Session ordinaire 2023-2024,...,0.34,0.799,0.000,1.0,2025-06-24,2024-06-06 09:00:00,SOC-A,SOC-A,Je laisse de côté la question de la cohérence ...,True
424503,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,0.10,0.958,0.000,1.0,2025-06-24,2024-06-06 15:00:00,DEM,DEM,"Monsieur Mournet, je n’ai pas le sentiment que...",True
424515,CRSANR5L16S2024O1N231,RUANR5L16S2024IDS28424,SCR5A2024O1,20240606150000000,jeudi 06 juin 2024,2,231,AN,16,Session ordinaire 2023-2024,...,0.73,0.973,0.000,1.0,2025-06-24,2024-06-06 15:00:00,RN,RN,"Monsieur le rapporteur général, je tiens à vou...",True
424956,CRSANR5L16S2024O1N233,RUANR5L16S2024IDS28426,SCR5A2024O1,20240607100000000,vendredi 07 juin 2024,1,233,AN,16,Session ordinaire 2023-2024,...,0.34,0.799,0.000,1.0,2025-06-24,2024-06-07 10:00:00,SOC-A,SOC-A,Je suis bouleversé par ce que nous sommes en t...,True


In [ ]:
df_match.to_csv(
    "../data/interim/df_repu.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

## Cf trace version matthias pour retrouver les noms de fichier
# # Exporter fichier uniquement avec Rep pour valeur absolues
# df_match.to_csv(
#     "../data/interim/df_regroup_repu_absolu.csv",
#     index=False,
#     # quoting=csv.QUOTE_ALL,  # not needed anymore ?
# )

In [ ]:
# Rajout en test d'une autre colonne avec le nombre de fois où la République apparait
# À tester/voir si fonctionne bien mais en tout cas absence de cas avec false et au moins 1


def count_lexical_outside_excl(text):
    if pd.isna(text):
        return 0

    # Trouver les positions des expressions exclues
    excl_positions = []
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
    )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Compter les occurrences valides
    count = 0
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            count += 1

    return count

In [50]:
df["nombre_correspondances"] = df["texte"].apply(count_lexical_outside_excl)

In [ ]:
# # verif criture/lecture ok
# print("df_match shape:", df_match.shape)

# df_test = pd.read_csv("../data/interim/df_repu.csv", low_memory=False)

# print("df_test shape (après export import): ", df_test.shape)